###  📥 PART :- B Data Acquisition 

#### 3️⃣ Import datasets from multiple scources :- 

- 📂 Load CSV files 
- 📑 Parse JSON files 
- 🗄️ Fetch records from SQL 
- 🌐 Fetch data from a dummy API 

In [1]:
import pandas as pd

transactions_df = pd.read_csv("customer_credit_risk_main_transactions_1000.csv")

In [2]:
transactions_df.head()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag
0,100001,61.0,Male,East,Secondary,Salaried,417788.81,477130.66,Home,389.0,0
1,100002,22.0,Male,South,Graduate,Salaried,641303.35,642665.38,Other,729.0,1
2,100003,38.0,Female,North,Graduate,Salaried,1160465.40,519796.55,Education,520.0,0
3,100004,27.0,Male,North,Secondary,Salaried,793688.41,99556.65,Business,770.0,0
4,100005,26.0,Female,East,Graduate,Self-Employed,286312.57,8988800.09,Car,596.0,0


In [3]:
transactions_df.shape

(1000, 11)

In [4]:
import pandas as pd

metadata_df = pd.read_json("customer_metadata_1000.json")

In [5]:
metadata_df.head()

,customer_id,spending_ratio,join_date
0,100001,15.02,2024-05-05
1,100002,27.04,2016-03-03
2,100003,71.28,2013-02-17
3,100004,71.66,2025-03-06
4,100005,12.82,2022-03-26


In [6]:
metadata_df.shape

(1000, 3)

In [7]:
import sqlite3
import random

conn = sqlite3.connect("loan_repayment_history.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE loan_repayment_history (
    customer_id INTEGER PRIMARY KEY,
    repayment_history INTEGER,
    transaction_count INTEGER
)
""")
rows = []
random.seed(42)

for i in range(1000):
    customer_id = 100001 + i
    repayment_history = random.randint(0, 12)
    transaction_count = random.randint(5, 300)

    if random.random() < 0.05:
        repayment_history = None
    if random.random() < 0.04:
        transaction_count = None

    if repayment_history is not None and random.random() < 0.01:
        repayment_history = random.randint(18, 35)   
    if transaction_count is not None and random.random() < 0.01:
        transaction_count = random.randint(800, 1200)  

    rows.append((customer_id, repayment_history, transaction_count))

cursor.executemany("""
INSERT INTO loan_repayment_history
VALUES (?, ?, ?)
""", rows)

conn.commit()
conn.close()

print("Database Created Successfully ✅")

Database Created Successfully ✅


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("loan_repayment_history.db")

repayment_df = pd.read_sql_query(
    "SELECT * FROM loan_repayment_history",
    conn
)

conn.close()

print(repayment_df.isna().sum())
repayment_df.head()

customer_id           0
repayment_history    50
transaction_count    37
dtype: int64


,customer_id,repayment_history,transaction_count
0,100001,NaN,62.0
1,100002,11.0,57.0
2,100003,0.0,52.0
3,100004,10.0,284.0
4,100005,12.0,86.0


In [9]:
import requests
import pandas as pd

url = "https://api.worldbank.org/v2/country/IND/indicator/FP.CPI.TOTL.ZG?format=json"

response = requests.get(url)

data = response.json()[1]

api_df = pd.DataFrame(data)

api_df = api_df[['country', 'date', 'value']]

api_df.columns = ['country', 'year', 'inflation_rate']

api_df.head()

,country,year,inflation_rate
0,"{'id': 'IN', 'value': 'India'}",2025,2.398850
1,"{'id': 'IN', 'value': 'India'}",2024,4.953036
2,"{'id': 'IN', 'value': 'India'}",2023,5.649143
3,"{'id': 'IN', 'value': 'India'}",2022,6.699034
4,"{'id': 'IN', 'value': 'India'}",2021,5.131407


In [10]:
import pandas as pd

# Merge CSV + JSON + SQL
final_df = (
    transactions_df
    .merge(metadata_df, on="customer_id", how="left")
    .merge(repayment_df, on="customer_id", how="left")
)

print(final_df.shape)
display(final_df.head())

(1000, 15)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag,spending_ratio,join_date,repayment_history,transaction_count
0,100001,61.0,Male,East,Secondary,Salaried,417788.81,477130.66,Home,389.0,0,15.02,2024-05-05,NaN,62.0
1,100002,22.0,Male,South,Graduate,Salaried,641303.35,642665.38,Other,729.0,1,27.04,2016-03-03,11.0,57.0
2,100003,38.0,Female,North,Graduate,Salaried,1160465.40,519796.55,Education,520.0,0,71.28,2013-02-17,0.0,52.0
3,100004,27.0,Male,North,Secondary,Salaried,793688.41,99556.65,Business,770.0,0,71.66,2025-03-06,10.0,284.0
4,100005,26.0,Female,East,Graduate,Self-Employed,286312.57,8988800.09,Car,596.0,0,12.82,2022-03-26,12.0,86.0


In [11]:
final_df.tail()

,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag,spending_ratio,join_date,repayment_history,transaction_count
995,100996,25.0,Male,West,Graduate,Salaried,1928566.09,1188118.57,Home,620.0,0,70.48,2015-12-07,2.0,83.0
996,100997,46.0,Male,West,Graduate,Self-Employed,2292294.62,2906712.86,Home,405.0,1,NaN,2019-01-17,10.0,230.0
997,100998,21.0,Male,East,Post-Graduate,Self-Employed,705111.71,138691.59,Other,677.0,0,9.60,2018-12-26,0.0,140.0
998,100999,21.0,Female,North,Secondary,Self-Employed,437064.35,NaN,Other,376.0,1,22.52,2022-08-08,4.0,248.0
999,101000,45.0,Male,South,Post-Graduate,Salaried,458171.29,253496.97,Business,451.0,0,30.97,2022-05-30,11.0,204.0


### 🧹 PART C : Data Understanding & Cleaning 

#### 4️⃣ Explore the dataset using pandas :- info() , describe() .

In [12]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   int64  
 1   age                950 non-null    float64
 2   gender             970 non-null    object 
 3   region             1000 non-null   object 
 4   education_level    1000 non-null   object 
 5   employment_type    960 non-null    object 
 6   annual_income      940 non-null    float64
 7   loan_amount        950 non-null    float64
 8   loan_purpose       1000 non-null   object 
 9   credit_score       950 non-null    float64
 10  default_flag       1000 non-null   int64  
 11  spending_ratio     950 non-null    float64
 12  join_date          1000 non-null   object 
 13  repayment_history  950 non-null    float64
 14  transaction_count  963 non-null    float64
dtypes: float64(7), int64(2), object(6)
memory usage: 117.3+ KB


In [13]:
final_df.describe()

,customer_id,age,annual_income,loan_amount,credit_score,default_flag,spending_ratio,repayment_history,transaction_count
count,1000.000000,950.000000,9.400000e+02,9.500000e+02,950.000000,1000.000000,950.000000,950.000000,963.000000
mean,100500.500000,43.346316,1.511707e+06,1.148060e+06,571.732632,0.218000,53.260221,6.116842,160.447560
std,288.819436,13.157515,1.575736e+06,1.184971e+06,164.485365,0.413094,34.989425,4.194938,110.160713
min,100001.000000,21.000000,1.826705e+05,5.105837e+04,51.000000,0.000000,5.260000,0.000000,5.000000
25%,100250.750000,32.000000,7.369092e+05,3.692860e+05,428.000000,0.000000,29.055000,3.000000,80.000000
50%,100500.500000,44.000000,1.355833e+06,8.135518e+05,568.500000,0.000000,50.715000,6.000000,157.000000
75%,100750.250000,55.000000,1.933494e+06,1.623187e+06,715.750000,0.000000,73.297500,9.000000,231.000000
max,101000.000000,65.000000,1.497379e+07,9.271784e+06,980.000000,1.000000,347.840000,35.000000,1180.000000


In [14]:
print("---------All Missing Values---------")
final_df.isnull().sum()

---------All Missing Values---------


customer_id           0
age                  50
gender               30
region                0
education_level       0
employment_type      40
annual_income        60
loan_amount          50
loan_purpose          0
credit_score         50
default_flag          0
spending_ratio       50
join_date             0
repayment_history    50
transaction_count    37
dtype: int64

#### 5️⃣ Perform Pandas Profiling to generate a data quality report .

In [15]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    final_df,
    title="Customer Credit Risk Data Quality Report",
    explorative=True
)

profile.to_file("customer_credit_risk_data_quality_report.html")


c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\admin\AppData\Local\Temp\ipykernel_9628\3191434623.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
Export report to file: 100%|██████████| 1/1 [00:00<00:00,  6.93it/s]


### 🧩 Handle missing data with :- 

### Simple Imputer (Numerical: Mean/Median)

In [16]:
from sklearn.impute import SimpleImputer

mean_df = final_df.copy()

num_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "spending_ratio",
    "repayment_history",
    "transaction_count"
]

mean_imputer = SimpleImputer(strategy="mean")

mean_df[num_cols] = mean_imputer.fit_transform(mean_df[num_cols])

print("Mean Imputation Completed Successfully.")

Mean Imputation Completed Successfully.


In [17]:
print("Missing Values After Mean Imputation:\n")
print(mean_df[num_cols].isnull().sum())

Missing Values After Mean Imputation:

age                  0
annual_income        0
loan_amount          0
credit_score         0
spending_ratio       0
repayment_history    0
transaction_count    0
dtype: int64


#####  🎯 Insights

- Mean imputation fills every numeric gap: the post-imputation null count is **0 for all numeric columns**.
- It is fast and keeps the column mean unchanged, but it **shrinks variance** and is distorted by the extreme incomes seen in `describe()`.

### Simple Imputer (categorical: most frequent).

In [18]:
cat_cols = [
    "gender",
    "employment_type"
]

print("Missing Values Before Imputation:\n")
print(final_df[cat_cols].isnull().sum())

Missing Values Before Imputation:

gender             30
employment_type    40
dtype: int64


In [19]:
from sklearn.impute import SimpleImputer

cat_imputed_df = final_df.copy()

cat_imputer = SimpleImputer(strategy="most_frequent")

cat_imputed_df[cat_cols] = cat_imputer.fit_transform(cat_imputed_df[cat_cols])

print("Categorical Imputation Completed Successfully.")

print("Missing Values After Imputation:\n")
print(cat_imputed_df[cat_cols].isnull().sum())

Categorical Imputation Completed Successfully.
Missing Values After Imputation:

gender             0
employment_type    0
dtype: int64


##### 🎯 Insights

- `SimpleImputer(strategy='most_frequent')` clears all missing `gender` and `employment_type` values.
- Being sklearn-based, it can be dropped straight into a pipeline/ColumnTransformer, unlike ad-hoc fills.
- The trade-off is that it **amplifies the majority category**, worsening the existing class imbalance.

### Most Frequent Category Imputation.

In [20]:
cat_cols = [
    "gender",
    "employment_type"
]

print(final_df[cat_cols].isnull().sum())

mfci_df = final_df.copy()

for col in cat_cols:
    mfci_df[col] = mfci_df[col].fillna(
        mfci_df[col].mode()[0]
    )

print("Most Frequent Category Imputation Completed Successfully.")
print(mfci_df[cat_cols].isnull().sum())

gender             30
employment_type    40
dtype: int64
Most Frequent Category Imputation Completed Successfully.
gender             0
employment_type    0
dtype: int64


##### 🎯 Insights

- The manual `mode()` fill reproduces the sklearn result exactly, confirming both approaches are equivalent.
- Pandas gives more transparency per column, while sklearn gives reusability inside pipelines.
- Either way, missingness in the categorical block is fully resolved.

### Missing Indicator + Random Sample Imputation.

In [21]:
import numpy as np

random_sample_df = final_df.copy()

random_sample_df["annual_income_missing"] = random_sample_df["annual_income"].isnull().astype(int)

random_values = random_sample_df["annual_income"].dropna().sample(
    random_sample_df["annual_income"].isnull().sum(),
    random_state=42,
    replace=True
)

random_sample_df.loc[
    random_sample_df["annual_income"].isnull(),
    "annual_income"
] = random_values.values

print("Missing Indicator + Random Sample Imputation Completed Successfully.")

Missing Indicator + Random Sample Imputation Completed Successfully.


In [22]:
print("-----------Verify-----------") 
print(random_sample_df[["annual_income", "annual_income_missing"]].head())
print("-----------Check Missing Values After Imputation-----------")
print(random_sample_df[["annual_income", "annual_income_missing"]].isnull().sum())
print("-----------Check Missing Indicator-----------")
random_sample_df["annual_income_missing"].value_counts()

-----------Verify-----------
   annual_income  annual_income_missing
0      417788.81                      0
1      641303.35                      0
2     1160465.40                      0
3      793688.41                      0
4      286312.57                      0
-----------Check Missing Values After Imputation-----------
annual_income            0
annual_income_missing    0
dtype: int64
-----------Check Missing Indicator-----------


annual_income_missing
0    940
1     60
Name: count, dtype: int64

##### 🎯 Insights

- Random sample imputation fills `annual_income` using values drawn from the observed distribution, so **skew and variance are preserved** far better than with mean imputation.
- The value counts show how many records were flagged, and the null count is now zero.

### ⚠️ Part D: Outlier Handling

### 7️⃣ Detect and treat outliers using:

### 📈 Z-score Method.

In [23]:
import numpy as np
from scipy.stats import zscore

zscore_df = final_df.copy()

num_cols = [
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count"
]

z_scores = np.abs(zscore(zscore_df[num_cols], nan_policy="omit"))

outliers = (z_scores > 3)

print("Outliers Detected:")
print(pd.DataFrame(outliers, columns=num_cols).sum())

zscore_df = zscore_df[(z_scores < 3).all(axis=1)]

print("\nOriginal Shape :", final_df.shape)
print("After Removing Outliers :", zscore_df.shape)

zscore_df.head()

Outliers Detected:
annual_income        15
loan_amount          15
credit_score          1
repayment_history     8
transaction_count     6
dtype: int64

Original Shape : (1000, 15)
After Removing Outliers : (745, 15)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag,spending_ratio,join_date,repayment_history,transaction_count
1,100002,22.0,Male,South,Graduate,Salaried,641303.35,642665.38,Other,729.0,1,27.04,2016-03-03,11.0,57.0
2,100003,38.0,Female,North,Graduate,Salaried,1160465.40,519796.55,Education,520.0,0,71.28,2013-02-17,0.0,52.0
3,100004,27.0,Male,North,Secondary,Salaried,793688.41,99556.65,Business,770.0,0,71.66,2025-03-06,10.0,284.0
5,100006,27.0,Male,West,Graduate,Salaried,1038819.84,365984.74,Home,573.0,0,7.86,2019-06-19,5.0,57.0
7,100008,35.0,Female,East,Secondary,Salaried,2298475.81,1977115.44,Business,622.0,0,59.18,2021-05-02,8.0,155.0


##### 🎯 Insights

- The Z-score rule (|z| > 3) flags a small number of extreme records, mostly in `annual_income` and `loan_amount`.
- Because it relies on mean and standard deviation, it is itself **pulled by the outliers**, so it under-detects in heavily skewed columns.
- It works well for `credit_score` and the behavioural counts, which are closer to symmetric.

###  📉 IQR Method.

In [24]:
import pandas as pd

iqr_df = final_df.copy()

num_cols = [
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count"
]

print("Original Shape:", iqr_df.shape)

outlier_count = {}

for col in num_cols:
    Q1 = iqr_df[col].quantile(0.25)
    Q3 = iqr_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count[col] = ((iqr_df[col] < lower_bound) |(iqr_df[col] > upper_bound)).sum()

    iqr_df = iqr_df[(iqr_df[col] >= lower_bound) &(iqr_df[col] <= upper_bound)]

print("\nOutliers Detected:")
print(pd.Series(outlier_count))

print("\nShape After Removing Outliers:", iqr_df.shape)

iqr_df.head()

Original Shape: (1000, 15)

Outliers Detected:
annual_income        15
loan_amount          13
credit_score          0
repayment_history     6
transaction_count     6
dtype: int64

Shape After Removing Outliers: (744, 15)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag,spending_ratio,join_date,repayment_history,transaction_count
1,100002,22.0,Male,South,Graduate,Salaried,641303.35,642665.38,Other,729.0,1,27.04,2016-03-03,11.0,57.0
2,100003,38.0,Female,North,Graduate,Salaried,1160465.40,519796.55,Education,520.0,0,71.28,2013-02-17,0.0,52.0
3,100004,27.0,Male,North,Secondary,Salaried,793688.41,99556.65,Business,770.0,0,71.66,2025-03-06,10.0,284.0
5,100006,27.0,Male,West,Graduate,Salaried,1038819.84,365984.74,Home,573.0,0,7.86,2019-06-19,5.0,57.0
7,100008,35.0,Female,East,Secondary,Salaried,2298475.81,1977115.44,Business,622.0,0,59.18,2021-05-02,8.0,155.0


##### 🎯 Insights

- The IQR rule uses quartiles, making it **robust to skew**, and it flags noticeably more extreme rows than the Z-score method.
- `repayment_history` and `transaction_count` are near-uniform, so they contribute almost no IQR outliers.

### 📊 Percentile Method.

In [25]:
percentile_df = final_df.copy()

num_cols = [
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count"
]

print("Original Shape:", percentile_df.shape)

for col in num_cols:

    lower = percentile_df[col].quantile(0.01)
    upper = percentile_df[col].quantile(0.99)

    percentile_df = percentile_df[
        (percentile_df[col] >= lower) &
        (percentile_df[col] <= upper)
    ]

print("Shape After Removing Outliers:", percentile_df.shape)

percentile_df.head()

Original Shape: (1000, 15)
Shape After Removing Outliers: (714, 15)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag,spending_ratio,join_date,repayment_history,transaction_count
1,100002,22.0,Male,South,Graduate,Salaried,641303.35,642665.38,Other,729.0,1,27.04,2016-03-03,11.0,57.0
2,100003,38.0,Female,North,Graduate,Salaried,1160465.40,519796.55,Education,520.0,0,71.28,2013-02-17,0.0,52.0
3,100004,27.0,Male,North,Secondary,Salaried,793688.41,99556.65,Business,770.0,0,71.66,2025-03-06,10.0,284.0
5,100006,27.0,Male,West,Graduate,Salaried,1038819.84,365984.74,Home,573.0,0,7.86,2019-06-19,5.0,57.0
7,100008,35.0,Female,East,Secondary,Salaried,2298475.81,1977115.44,Business,622.0,0,59.18,2021-05-02,8.0,155.0


##### 🎯 Insights

- Percentile trimming at the 1st/99th bounds gives **explicit control over how much data is dropped** rather than letting the statistics decide.
- The reduced shape confirms only the extreme tails were cut while the bulk of the distribution is intact.

### ✂️ Winsorization Technique.

In [26]:
winsor_df = final_df.copy()

num_cols = [
    "annual_income",
    "loan_amount",
    "credit_score",
    "repayment_history",
    "transaction_count"
]

for col in num_cols:

    lower = winsor_df[col].quantile(0.05)
    upper = winsor_df[col].quantile(0.95)

    winsor_df[col] = winsor_df[col].clip(lower, upper)

print("Winsorization Applied Successfully!")

winsor_df.head()

Winsorization Applied Successfully!


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,default_flag,spending_ratio,join_date,repayment_history,transaction_count
0,100001,61.0,Male,East,Secondary,Salaried,4.177888e+05,4.771307e+05,Home,389.0,0,15.02,2024-05-05,NaN,62.0
1,100002,22.0,Male,South,Graduate,Salaried,6.413033e+05,6.426654e+05,Other,729.0,1,27.04,2016-03-03,11.0,57.0
2,100003,38.0,Female,North,Graduate,Salaried,1.160465e+06,5.197965e+05,Education,520.0,0,71.28,2013-02-17,0.0,52.0
3,100004,27.0,Male,North,Secondary,Salaried,7.936884e+05,1.047665e+05,Business,770.0,0,71.66,2025-03-06,10.0,284.0
4,100005,26.0,Female,East,Graduate,Self-Employed,3.051711e+05,2.898341e+06,Car,596.0,0,12.82,2022-03-26,12.0,86.0


##### 🎯 Insights

- Winsorization at the 5th/95th percentiles **caps** extreme values instead of deleting rows, so all 1000 customers are retained.
- Maxima and minima are pulled inwards, which tames the influence of extreme incomes and loans on scaling and models.

### 🧬 Part E: Feature Engineering

### 8️⃣ Handle variable types:

#### 🔄 Mixed Variables (numeric + categorical).

In [27]:
numerical_cols = final_df.select_dtypes(include=["int64", "float64"]).columns

categorical_cols = final_df.select_dtypes(include=["object"]).columns

print("Numerical Columns:")
print(list(numerical_cols))

print("\nCategorical Columns:")
print(list(categorical_cols))

Numerical Columns:
['customer_id', 'age', 'annual_income', 'loan_amount', 'credit_score', 'default_flag', 'spending_ratio', 'repayment_history', 'transaction_count']

Categorical Columns:
['gender', 'region', 'education_level', 'employment_type', 'loan_purpose', 'join_date']


##### 🎯 Insights

- The split cleanly separates numeric columns from `object` categoricals, which is the foundation for a ColumnTransformer.
- It also exposes mixed-type handling: `gender` is categorical, `age` numeric and `spending_ratio` numeric but skewed.

#### 📅 Date & Time variables → extract Year, Month, Day, Weekday.

In [28]:
final_df["join_date"] = pd.to_datetime(final_df["join_date"])

final_df["join_year"] = final_df["join_date"].dt.year
final_df["join_month"] = final_df["join_date"].dt.month
final_df["join_day"] = final_df["join_date"].dt.day
final_df["join_weekday"] = final_df["join_date"].dt.day_name()

final_df[[
    "join_date",
    "join_year",
    "join_month",
    "join_day",
    "join_weekday"
]].head()

,join_date,join_year,join_month,join_day,join_weekday
0,2024-05-05,2024,5,5,Sunday
1,2016-03-03,2016,3,3,Thursday
2,2013-02-17,2013,2,17,Sunday
3,2025-03-06,2025,3,6,Thursday
4,2022-03-26,2022,3,26,Saturday


##### 🎯 Insights

- Converting `join_date` to datetime unlocks `join_year`, `join_month`, `join_day` and `join_weekday`.
- A single opaque date becomes **four usable features**, allowing tenure, seasonality and weekday effects to be learned.
- The extracted values line up correctly with the original dates in the preview.

### 9️⃣ Encoding categorical variables:

#### 🔠 Ordinal Encoding (education levels).

In [29]:
ordinal_df = final_df.copy()

print("Before Encoding:")
display(ordinal_df[["education_level"]].head())

education_order = {
    "Primary": 0,
    "Secondary": 1,
    "Graduate": 2,
    "Post-Graduate": 3
}

ordinal_df["education_level"] = ordinal_df["education_level"].map(education_order)

print("\nAfter Encoding:")
display(ordinal_df[["education_level"]].head())

Before Encoding:


,education_level
0,Secondary
1,Graduate
2,Graduate
3,Secondary
4,Graduate



After Encoding:


,education_level
0,1
1,2
2,2
3,1
4,2


##### 🎯 Insights

- Ordinal encoding maps education to **0–3 in a meaningful order** (Primary < Secondary < Graduate < Post-Graduate).
- The before/after preview confirms the ranking is preserved, which one-hot encoding would have thrown away.
- Correct choice here because education is genuinely ordered.

#### 🔠 Label Encoding (binary features).

In [30]:
from sklearn.preprocessing import LabelEncoder

label_df = final_df.copy()

print("Before Encoding:")
display(label_df[["gender"]].head())

label_encoder = LabelEncoder()

label_df["gender"] = label_encoder.fit_transform(label_df["gender"])

print("\nAfter Encoding:")
display(label_df[["gender"]].head())

mapping = dict(zip(label_encoder.classes_,label_encoder.transform(label_encoder.classes_)))

print("\nLabel Encoding Mapping:")
print(mapping)

Before Encoding:


,gender
0,Male
1,Male
2,Female
3,Male
4,Female



After Encoding:


,gender
0,1
1,1
2,0
3,1
4,0



Label Encoding Mapping:
{'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2), nan: np.int64(3)}


##### 🎯 Insights

- Label encoding converts `gender` into compact integers, and the printed mapping documents exactly which class became which code.
- Appropriate because the variable is essentially **binary**, so no false ordering is introduced.
- The mapping must be reused at inference time to avoid label drift.

#### 🔠 One-Hot Encoding (regions, loan purpose).

In [31]:
onehot_df = final_df.copy()

print("Before Encoding:")
display(onehot_df[["region", "loan_purpose"]].head())

onehot_df = pd.get_dummies(
    onehot_df,
    columns=["region", "loan_purpose"],
    dtype=int)

onehot_df.filter(regex="region_|loan_purpose_").head()

Before Encoding:


,region,loan_purpose
0,East,Home
1,South,Other
2,North,Education
3,North,Business
4,East,Car


,region_East,region_North,region_South,region_West,loan_purpose_Business,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Other
0,1,0,0,0,0,0,0,1,0
1,0,0,1,0,0,0,0,0,1
2,0,1,0,0,0,0,1,0,0
3,0,1,0,0,1,0,0,0,0
4,1,0,0,0,0,1,0,0,0


##### 🎯 Insights

- One-hot encoding expands `region` and `loan_purpose` into indicator columns (4 regions + 5 purposes).
- These are **nominal** variables, so one-hot avoids inventing an order that does not exist.
- Cost is extra width, which is acceptable here given the low cardinality.

### 🔟 Encoding numerical features:

#### 📶 Binning (discretize income into groups).

In [32]:
binning_df = final_df.copy()

binning_df["income_group"] = pd.cut(
    binning_df["annual_income"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

display(
    binning_df[
        ["annual_income", "income_group"]
    ].head(10)
)

,annual_income,income_group
0,417788.81,Low
1,641303.35,Low
2,1160465.40,Low
3,793688.41,Low
4,286312.57,Low
5,1038819.84,Low
6,2475694.17,Low
7,2298475.81,Low
8,1908374.44,Low
9,289309.99,Low


In [33]:
binning_df["income_group"].value_counts()

income_group
Low       925
High       14
Medium      1
Name: count, dtype: int64

##### 🎯 Insights

- Equal-width binning turns continuous income into Low/Medium/High groups, making the variable easier to interpret.
- The value counts reveal **unbalanced bins** — a direct consequence of the skewed income distribution.
- Useful for reporting, but quantile binning gives more even groups for modeling.

#### 📶 Binarization (flag if > threshold).

In [34]:
from sklearn.preprocessing import Binarizer

binarization_df = final_df.copy()

binarization_df["credit_score"] = (
    binarization_df["credit_score"]
    .fillna(binarization_df["credit_score"].median())
)

binarizer = Binarizer(threshold=700)

binarization_df["credit_score_flag"] = binarizer.fit_transform(
    binarization_df[["credit_score"]]
).astype(int)

display(
    binarization_df[
        ["credit_score", "credit_score_flag"]
    ].head(10)
)

,credit_score,credit_score_flag
0,389.0,0
1,729.0,1
2,520.0,0
3,770.0,1
4,596.0,0
5,573.0,0
6,524.0,0
7,622.0,0
8,738.0,1
9,463.0,0


##### 🎯 Insights

- Binarization at a threshold of 700 creates a crisp `credit_score_flag` separating prime from sub-prime customers.
- The median fill first ensures the binarizer receives no missing values.
- It compresses a rich variable into one bit, but that bit maps neatly to real lending policy.

#### 📶 Quantile Binning.

In [35]:
import pandas as pd

quantile_df = final_df.copy()

print("Before Quantile Binning:")
display(quantile_df[["transaction_count"]].head())

quantile_df["transaction_count_quantile"] = pd.qcut(
    quantile_df["transaction_count"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print("\nAfter Quantile Binning:")
display(
    quantile_df[
        ["transaction_count", "transaction_count_quantile"]
    ].head(10)
)

Before Quantile Binning:


,transaction_count
0,62.0
1,57.0
2,52.0
3,284.0
4,86.0



After Quantile Binning:


,transaction_count,transaction_count_quantile
0,62.0,Q1
1,57.0,Q1
2,52.0,Q1
3,284.0,Q4
4,86.0,Q2
5,57.0,Q1
6,27.0,Q1
7,155.0,Q2
8,40.0,Q1
9,124.0,Q2


##### 🎯 Insights

- Quantile binning with `qcut` produces **four roughly equal-sized groups** (Q1–Q4) of `transaction_count`.
- Unlike equal-width bins, group sizes are balanced by construction, which helps models and comparisons.
- The before/after view shows low counts landing in Q1 and the most active customers in Q4.

#### 📶 K-Means Binning.

In [36]:
from sklearn.cluster import KMeans

kmeans_df = final_df.copy()

kmeans_df["transaction_count"] = kmeans_df["transaction_count"].fillna(
    kmeans_df["transaction_count"].median()
)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

kmeans_df["kmeans_cluster"] = kmeans.fit_predict(
    kmeans_df[["transaction_count"]]
)

cluster_order = (
    kmeans_df.groupby("kmeans_cluster")["transaction_count"]
    .mean()
    .sort_values()
    .index
)

mapping = {cluster: i for i, cluster in enumerate(cluster_order)}

kmeans_df["transaction_count_kmeans"] = (
    kmeans_df["kmeans_cluster"].map(mapping)
)

display(
    kmeans_df[
        ["transaction_count", "transaction_count_kmeans"]
    ].head(10)
)

,transaction_count,transaction_count_kmeans
0,62.0,0
1,57.0,0
2,52.0,0
3,284.0,2
4,86.0,0
5,57.0,0
6,27.0,0
7,155.0,1
8,40.0,0
9,124.0,1


##### 🎯 Insights

- K-Means binning lets the **data itself decide** the cut points instead of imposing fixed widths or quantiles.
- Relabelling clusters by mean transaction count makes the cluster IDs ordered and interpretable.
- Requires imputation and a fixed `random_state` to stay reproducible.

### Part F: Feature Scaling

### 1️⃣1️⃣ Apply multiple scaling methods:

### 📏 Standardization (Z-score scaling).

In [37]:
from sklearn.preprocessing import StandardScaler

standard_df = final_df.copy()

standard_df["annual_income"] = standard_df["annual_income"].fillna(
    standard_df["annual_income"].median()
)
standard_df["loan_amount"] = standard_df["loan_amount"].fillna(
    standard_df["loan_amount"].median()
)

print("Before Standardization:")
display(standard_df[["annual_income", "loan_amount"]].head())

scaler = StandardScaler()

standard_df[["annual_income", "loan_amount"]] = scaler.fit_transform(
    standard_df[["annual_income", "loan_amount"]]
)

print("\nAfter Standardization:")
display(standard_df[["annual_income", "loan_amount"]].head())

Before Standardization:


,annual_income,loan_amount
0,417788.81,477130.66
1,641303.35,642665.38
2,1160465.40,519796.55
3,793688.41,99556.65
4,286312.57,8988800.09



After Standardization:


,annual_income,loan_amount
0,-0.710088,-0.565598
1,-0.563748,-0.422484
2,-0.223842,-0.528711
3,-0.463978,-0.892034
4,-0.796168,6.793244


##### 🎯 Insights

- Standardization recentres `annual_income` and `loan_amount` to mean 0 and unit variance, so the two columns become directly comparable.
- Essential for distance- and gradient-based models (KNN, SVM, logistic regression) where raw rupee magnitudes would dominate.
- It rescales but does **not** remove skew or outliers — those still need Part D and Part G treatment.

###  📐 Normalization.

In [38]:
from sklearn.preprocessing import Normalizer

normalization_df = final_df.copy()

normalization_df["annual_income"] = normalization_df["annual_income"].fillna(
    normalization_df["annual_income"].median())
normalization_df["loan_amount"] = normalization_df["loan_amount"].fillna(
    normalization_df["loan_amount"].median())

print("Before Normalization:")
display(normalization_df[["annual_income", "loan_amount"]].head())

normalizer = Normalizer()

normalization_df[["annual_income", "loan_amount"]] = normalizer.fit_transform(
    normalization_df[["annual_income", "loan_amount"]])

print("\nAfter Normalization:")
display(normalization_df[["annual_income", "loan_amount"]].head())

Before Normalization:


,annual_income,loan_amount
0,417788.81,477130.66
1,641303.35,642665.38
2,1160465.40,519796.55
3,793688.41,99556.65
4,286312.57,8988800.09



After Normalization:


,annual_income,loan_amount
0,0.658772,0.752343
1,0.706356,0.707856
2,0.912630,0.408786
3,0.992225,0.124460
4,0.031836,0.999493


##### 🎯 Insights

- `Normalizer` works **row-wise**, scaling each customer's vector to unit norm rather than standardizing each column.
- Values now describe the customer's internal income/loan proportion, not their absolute level.

### 📊 Min-Max Scaling.

In [39]:
from sklearn.preprocessing import MinMaxScaler

minmax_df = final_df.copy()

numeric_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "spending_ratio",
    "repayment_history",
    "transaction_count"
]

minmax_df[numeric_cols] = minmax_df[numeric_cols].fillna(
    minmax_df[numeric_cols].median()
)

print("Before Min-Max Scaling:")
display(minmax_df[numeric_cols].head())

scaler = MinMaxScaler()

minmax_df[numeric_cols] = scaler.fit_transform(
    minmax_df[numeric_cols])

print("\nAfter Min-Max Scaling:")
display(minmax_df[numeric_cols].head())

Before Min-Max Scaling:


,age,annual_income,loan_amount,credit_score,spending_ratio,repayment_history,transaction_count
0,61.0,417788.81,477130.66,389.0,15.02,6.0,62.0
1,22.0,641303.35,642665.38,729.0,27.04,11.0,57.0
2,38.0,1160465.40,519796.55,520.0,71.28,0.0,52.0
3,27.0,793688.41,99556.65,770.0,71.66,10.0,284.0
4,26.0,286312.57,8988800.09,596.0,12.82,12.0,86.0



After Min-Max Scaling:


,age,annual_income,loan_amount,credit_score,spending_ratio,repayment_history,transaction_count
0,0.909091,0.015896,0.046208,0.363832,0.028490,0.171429,0.048511
1,0.022727,0.031007,0.064161,0.729817,0.063576,0.314286,0.044255
2,0.386364,0.066107,0.050835,0.504844,0.192714,0.000000,0.040000
3,0.136364,0.041310,0.005260,0.773950,0.193823,0.285714,0.237447
4,0.113636,0.007007,0.969310,0.586652,0.022068,0.342857,0.068936


##### 🎯 Insights

- Min-Max scaling compresses every numeric feature into a clean **[0, 1]** range, preserving the original distribution shape.
- Ideal for neural networks and any algorithm expecting bounded inputs.
- It is **very sensitive to outliers**, since a single extreme income squeezes all other values towards zero.

### ➕ MaxAbs Scaling.

In [40]:
from sklearn.preprocessing import MaxAbsScaler

maxabs_df = final_df.copy()

numeric_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "spending_ratio",
    "repayment_history",
    "transaction_count"
]

maxabs_df[numeric_cols] = maxabs_df[numeric_cols].fillna(
    maxabs_df[numeric_cols].median()
)

print("Before MaxAbs Scaling:")
display(maxabs_df[numeric_cols].head())

scaler = MaxAbsScaler()

maxabs_df[numeric_cols] = scaler.fit_transform(
    maxabs_df[numeric_cols]
)

print("\nAfter MaxAbs Scaling:")
display(maxabs_df[numeric_cols].head())

Before MaxAbs Scaling:


,age,annual_income,loan_amount,credit_score,spending_ratio,repayment_history,transaction_count
0,61.0,417788.81,477130.66,389.0,15.02,6.0,62.0
1,22.0,641303.35,642665.38,729.0,27.04,11.0,57.0
2,38.0,1160465.40,519796.55,520.0,71.28,0.0,52.0
3,27.0,793688.41,99556.65,770.0,71.66,10.0,284.0
4,26.0,286312.57,8988800.09,596.0,12.82,12.0,86.0



After MaxAbs Scaling:


,age,annual_income,loan_amount,credit_score,spending_ratio,repayment_history,transaction_count
0,0.938462,0.027901,0.051461,0.396939,0.043181,0.171429,0.052542
1,0.338462,0.042828,0.069314,0.743878,0.077737,0.314286,0.048305
2,0.584615,0.077500,0.056062,0.530612,0.204922,0.000000,0.044068
3,0.415385,0.053005,0.010738,0.785714,0.206014,0.285714,0.240678
4,0.400000,0.019121,0.969479,0.608163,0.036856,0.342857,0.072881


##### 🎯 Insights

- MaxAbs scaling divides by the maximum absolute value, mapping data into [-1, 1] while **preserving zeros and sparsity**.
- With all-positive columns here the result is effectively [0, 1], very close to Min-Max.

### 🛡️ Robust Scaling.

In [41]:
from sklearn.preprocessing import RobustScaler

robust_df = final_df.copy()

numeric_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "spending_ratio",
    "repayment_history",
    "transaction_count"
]

robust_df[numeric_cols] = robust_df[numeric_cols].fillna(
    robust_df[numeric_cols].median()
)

print("Before Robust Scaling:")
display(robust_df[numeric_cols].head())

scaler = RobustScaler()

robust_df[numeric_cols] = scaler.fit_transform(
    robust_df[numeric_cols]
)

print("\nAfter Robust Scaling:")
display(robust_df[numeric_cols].head())

Before Robust Scaling:


,age,annual_income,loan_amount,credit_score,spending_ratio,repayment_history,transaction_count
0,61.0,417788.81,477130.66,389.0,15.02,6.0,62.0
1,22.0,641303.35,642665.38,729.0,27.04,11.0,57.0
2,38.0,1160465.40,519796.55,520.0,71.28,0.0,52.0
3,27.0,793688.41,99556.65,770.0,71.66,10.0,284.0
4,26.0,286312.57,8988800.09,596.0,12.82,12.0,86.0



After Robust Scaling:


,age,annual_income,loan_amount,credit_score,spending_ratio,repayment_history,transaction_count
0,0.809524,-0.833212,-0.280245,-0.656307,-0.853488,0.000000,-0.659722
1,-1.047619,-0.634677,-0.142351,0.586837,-0.566083,0.833333,-0.694444
2,-0.285714,-0.173534,-0.244703,-0.177331,0.491721,-1.000000,-0.729167
3,-0.809524,-0.499322,-0.594771,0.736746,0.500807,0.666667,0.881944
4,-0.857143,-0.949995,6.810128,0.100548,-0.906091,1.000000,-0.493056


##### 🎯 Insights

- Robust scaling centres on the **median** and scales by the **IQR**, so extreme incomes and loans barely move the transformation.
- The scaled ranges stay tighter and better behaved than Min-Max on this skewed data.
- For this dataset it is the **most reliable scaler**, which is why the final pipeline adopts it.

### 🧠 Part G: Feature Construction & Transformation

### 1️⃣2️⃣ Apply transformations:

### 🔧  FunctionTransformer → log transform, reciprocal, square root.

In [42]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

transform_df = final_df.copy()

transform_df["spending_ratio"] = transform_df["spending_ratio"].fillna(
    transform_df["spending_ratio"].median()
)

print("Before Transformation:")
display(transform_df[["spending_ratio"]].head())

log_transformer = FunctionTransformer(np.log1p)
transform_df["log_spending_ratio"] = log_transformer.transform(
    transform_df[["spending_ratio"]]
)

reciprocal_transformer = FunctionTransformer(lambda x: 1 / (x + 1))
transform_df["reciprocal_spending_ratio"] = reciprocal_transformer.transform(
    transform_df[["spending_ratio"]]
)

sqrt_transformer = FunctionTransformer(np.sqrt)
transform_df["sqrt_spending_ratio"] = sqrt_transformer.transform(
    transform_df[["spending_ratio"]]
)

print("\nAfter Transformation:")
display(
    transform_df[["spending_ratio","log_spending_ratio","reciprocal_spending_ratio","sqrt_spending_ratio"]].head()
)

Before Transformation:


,spending_ratio
0,15.02
1,27.04
2,71.28
3,71.66
4,12.82



After Transformation:


,spending_ratio,log_spending_ratio,reciprocal_spending_ratio,sqrt_spending_ratio
0,15.02,2.773838,0.062422,3.875564
1,27.04,3.333632,0.035663,5.200000
2,71.28,4.280547,0.013835,8.442748
3,71.66,4.285791,0.013763,8.465223
4,12.82,2.626117,0.072359,3.580503


##### 🎯 Insights

- `FunctionTransformer` applies log, reciprocal and square-root transforms to `spending_ratio` without writing custom classes.
- `log1p` pulls in the long right tail most effectively, making the distribution far closer to normal.

### ⚡ PowerTransformer → Box-Cox and Yeo-Johnson.

In [43]:
from sklearn.preprocessing import PowerTransformer

power_df = final_df.copy()

power_df["loan_amount"] = power_df["loan_amount"].fillna(
    power_df["loan_amount"].median()
)

power_df["annual_income"] = power_df["annual_income"].fillna(
    power_df["annual_income"].median()
)

print("Before Transformation:")
display(power_df[["loan_amount", "annual_income"]].head())

boxcox = PowerTransformer(method="box-cox")
power_df["loan_amount_boxcox"] = boxcox.fit_transform(
    power_df[["loan_amount"]]
)

yeojohnson = PowerTransformer(method="yeo-johnson")
power_df["annual_income_yeojohnson"] = yeojohnson.fit_transform(
    power_df[["annual_income"]]
)

print("\nAfter Transformation:")
display(power_df[["loan_amount","loan_amount_boxcox","annual_income","annual_income_yeojohnson"]].head())

Before Transformation:


,loan_amount,annual_income
0,477130.66,417788.81
1,642665.38,641303.35
2,519796.55,1160465.40
3,99556.65,793688.41
4,8988800.09,286312.57



After Transformation:


,loan_amount,loan_amount_boxcox,annual_income,annual_income_yeojohnson
0,477130.66,-0.504679,417788.81,-1.461555
1,642665.38,-0.213292,641303.35,-0.876310
2,519796.55,-0.422266,1160465.40,-0.038014
3,99556.65,-1.833757,793688.41,-0.578814
4,8988800.09,3.053287,286312.57,-1.963841


##### 🎯 Insights

- `PowerTransformer` estimates the optimal lambda automatically, so Box-Cox / Yeo-Johnson beat hand-picked transforms.
- Box-Cox needs strictly positive values, while **Yeo-Johnson also handles zeros and negatives** — the safer general choice.
- Post-transform `loan_amount` and `annual_income` are far more symmetric, which benefits linear and distance-based models.

### 🧮 Column Transformer → apply different preprocessing steps to different columns.

In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

num_cols = ["annual_income", "loan_amount"]
cat_cols = ["region"]

ct = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(), cat_cols)
])

result = ct.fit_transform(final_df)

print(result[:5])
print(result.shape)
display(pd.DataFrame(result).head())

[[-0.69459612 -0.56649756  1.          0.          0.          0.        ]
 [-0.55267292 -0.42672882  0.          0.          1.          0.        ]
 [-0.22302484 -0.53047275  0.          1.          0.          0.        ]
 [-0.45591422 -0.88530105  0.          1.          0.          0.        ]
 [-0.77807851  6.62030508  1.          0.          0.          0.        ]]
(1000, 6)


,0,1,2,3,4,5
0,-0.694596,-0.566498,1.0,0.0,0.0,0.0
1,-0.552673,-0.426729,0.0,0.0,1.0,0.0
2,-0.223025,-0.530473,0.0,1.0,0.0,0.0
3,-0.455914,-0.885301,0.0,1.0,0.0,0.0
4,-0.778079,6.620305,1.0,0.0,0.0,0.0


##### 🎯 Insights

- `ColumnTransformer` applies StandardScaler to numeric columns and OneHotEncoder to `region` in a **single fitted object**.
- The output width equals scaled numerics plus region dummies, exactly as expected.
- This is the production-ready pattern: no leakage, no manual column bookkeeping, and it can be dropped into a Pipeline.

### 1️⃣3️⃣ Construct new features:

### 💰 Debt-to-Income ratio.

In [45]:
feature_df = final_df.copy()

feature_df["annual_income"] = feature_df["annual_income"].fillna(
    feature_df["annual_income"].median()
)

feature_df["loan_amount"] = feature_df["loan_amount"].fillna(
    feature_df["loan_amount"].median()
)

feature_df["debt_to_income_ratio"] = (
    feature_df["loan_amount"] / feature_df["annual_income"]
)

print("New Feature:")
display(
    feature_df[
        ["annual_income", "loan_amount", "debt_to_income_ratio"]
    ].head()
)

New Feature:


,annual_income,loan_amount,debt_to_income_ratio
0,417788.81,477130.66,1.142038
1,641303.35,642665.38,1.002124
2,1160465.40,519796.55,0.447921
3,793688.41,99556.65,0.125435
4,286312.57,8988800.09,31.395059


##### 🎯 Insights

- `debt_to_income_ratio = loan_amount / annual_income` compresses two columns into one **risk-dense** feature.
- Ratios above ~1 mark customers borrowing more than they earn annually — a classic default signal.
- Being scale-free, it is comparable across income brackets in a way the raw amounts are not.

### 💰 Average monthly transactions.

In [46]:
feature_df = final_df.copy()

feature_df["transaction_count"] = feature_df["transaction_count"].fillna(
    feature_df["transaction_count"].median()
)

feature_df["avg_monthly_transaction"] = (
    feature_df["transaction_count"] / 12
)

print("New Feature:")
display(
    feature_df[
        ["transaction_count", "avg_monthly_transaction"]
    ].head()
)

New Feature:


,transaction_count,avg_monthly_transaction
0,62.0,5.166667
1,57.0,4.750000
2,52.0,4.333333
3,284.0,23.666667
4,86.0,7.166667


##### 🎯 Insights

- `avg_monthly_transaction = transaction_count / 12` converts a raw total into an interpretable **monthly activity rate**.
- Normalising by time makes engagement comparable across customers and removes period-length ambiguity.
- Low activity combined with high debt-to-income is a particularly useful risk combination.

### 💰 Spending-to-Income ratio.

In [47]:
feature_df = final_df.copy()

feature_df["annual_income"] = feature_df["annual_income"].fillna(
    feature_df["annual_income"].median()
)

feature_df["spending_ratio"] = feature_df["spending_ratio"].fillna(
    feature_df["spending_ratio"].median()
)

feature_df["spending_to_income_ratio"] = (
    feature_df["spending_ratio"] / feature_df["annual_income"]
)

print("New Feature:")
display(
    feature_df[
        ["spending_ratio", "annual_income", "spending_to_income_ratio"]
    ].head()
)

New Feature:


,spending_ratio,annual_income,spending_to_income_ratio
0,15.02,417788.81,0.000036
1,27.04,641303.35,0.000042
2,71.28,1160465.40,0.000061
3,71.66,793688.41,0.000090
4,12.82,286312.57,0.000045


##### 🎯 Insights

- `spending_to_income_ratio` links behavioural spending back to earning capacity, capturing financial strain.
- Median fills on both inputs ensure the ratio is defined for every customer.
- Together with debt-to-income and average monthly transactions, this completes a compact set of **three engineered risk features**.

### 📊 Part H: Final Deliverable

### 1️⃣4️⃣ Provide a final cleaned and transformed dataset.



In [48]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, RobustScaler
from scipy.stats.mstats import winsorize

final_clean_df = final_df.copy()

In [49]:
# 1. Missing values
num_cols = ["age", "annual_income", "loan_amount", "credit_score",
            "spending_ratio", "repayment_history", "transaction_count"]
final_clean_df[num_cols] = KNNImputer(n_neighbors=5).fit_transform(final_clean_df[num_cols])

cat_cols = ["gender", "employment_type"]
final_clean_df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(final_clean_df[cat_cols])

# 2. Outliers -> Winsorize
for col in ["annual_income", "loan_amount", "credit_score", "repayment_history", "transaction_count"]:
    final_clean_df[col] = winsorize(final_clean_df[col], limits=[0.01, 0.01])

# 3. Encoding
final_clean_df["education_level"] = final_clean_df["education_level"].map(
    {"Primary": 0, "Secondary": 1, "Graduate": 2, "Post-Graduate": 3}
)
final_clean_df["gender"] = LabelEncoder().fit_transform(final_clean_df["gender"])
final_clean_df = pd.get_dummies(final_clean_df, columns=["region", "loan_purpose"], dtype=int)

# 4. Scaling
scale_cols = ["age", "annual_income", "loan_amount", "credit_score"]
final_clean_df[scale_cols] = RobustScaler().fit_transform(final_clean_df[scale_cols])

# 5. New features
final_clean_df["debt_to_income_ratio"] = final_clean_df["loan_amount"] / final_clean_df["annual_income"]
final_clean_df["avg_monthly_transaction"] = final_clean_df["transaction_count"] / 6
final_clean_df["spending_to_income_ratio"] = final_clean_df["spending_ratio"] / 100

print("Final dataset shape:", final_clean_df.shape)
print("Missing values left:", final_clean_df.isna().sum().sum())
final_clean_df.head()

Final dataset shape: (1000, 29)
Missing values left: 0


,customer_id,age,gender,education_level,employment_type,annual_income,loan_amount,credit_score,default_flag,spending_ratio,...,region_South,region_West,loan_purpose_Business,loan_purpose_Car,loan_purpose_Education,loan_purpose_Home,loan_purpose_Other,debt_to_income_ratio,avg_monthly_transaction,spending_to_income_ratio
0,100001,0.807601,1,1,Salaried,-0.830733,-0.314785,-0.679492,0,15.02,...,0,0,0,0,0,1,0,0.378924,10.333333,0.1502
1,100002,-1.045131,1,2,Salaried,-0.632360,-0.181714,0.554628,1,27.04,...,1,0,0,0,0,0,1,0.287359,9.500000,0.2704
2,100003,-0.285036,0,2,Salaried,-0.171594,-0.280486,-0.203993,0,71.28,...,0,0,0,0,1,0,0,1.634596,8.666667,0.7128
3,100004,-0.807601,1,1,Salaried,-0.497115,-0.618310,0.703448,0,71.66,...,0,0,1,0,0,0,0,1.243797,47.333333,0.7166
4,100005,-0.855107,0,2,Self-Employed,-0.947420,4.937994,0.071869,0,12.82,...,0,0,0,1,0,0,0,-5.212041,14.333333,0.1282


In [50]:
final_clean_df.to_csv("final_processed_dataset.csv", index=False)
print("Saved final_processed_dataset.csv")

Saved final_processed_dataset.csv


### 1️⃣5️⃣ Write a report summarizing :- 

- 🧩 Missing value strategies used and their effectiveness.
- ⚠️ Outlier handling results.
- 🔠 Encoding methods applied to categorical/numerical variables.
- ⚙️ Scaling/transformations applied and why.
- 💎 Newly engineered features and their usefulness.
- 📊 Final dataset shape and readiness for ML modeling.